In [3]:
import requests
import pandas as pd
import holidays
import os
import matplotlib.pyplot as plt

START = '2019-01-01'
END   = '2024-12-31'
OUTPUT = '../../data/raw/economic_hourly.csv'

In [ ]:
r = requests.get('https://fred.stlouisfed.org/graph/fredgraph.csv?id=DHHNGSP', timeout=30)
r.raise_for_status()

ng = pd.read_csv(
    pd.io.common.StringIO(r.text),
    parse_dates=['observation_date'],
    index_col='observation_date'
)
ng.columns = ['gas_price_mmbtu']
ng = ng.replace('.', float('nan'))
ng['gas_price_mmbtu'] = pd.to_numeric(ng['gas_price_mmbtu'], errors='coerce')
ng = ng.loc[START:END]

### 2. Build hourly datetime index and expand gas price

In [ ]:
hourly_idx = pd.date_range(start=START, end=END + ' 23:00', freq='h', tz='America/Los_Angeles')

df = pd.DataFrame(index=hourly_idx)
df.index.name = 'datetime'

ng.index = pd.DatetimeIndex(ng.index).tz_localize('America/Los_Angeles')
df['gas_price_mmbtu'] = ng['gas_price_mmbtu'].reindex(df.index, method='ffill')

### 3. Calendar features

In [ ]:
ca_holidays = holidays.country_holidays('US', subdiv='CA', years=range(2019, 2025))

dates = df.index.normalize()

df['is_holiday'] = dates.isin(ca_holidays).astype(int)
df['is_weekend']  = (df.index.dayofweek >= 5).astype(int)
df['hour_of_day'] = df.index.hour
df['day_of_week'] = df.index.dayofweek
df['month']       = df.index.month

In [ ]:
os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)
df.to_csv(OUTPUT)